In [3]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

# 1. Load and Preprocess Iris Dataset
iris = load_iris()
X = iris.data
y = iris.target.reshape(-1, 1)

encoder = OneHotEncoder(sparse_output=False)
y_encoded = encoder.fit_transform(y)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded, test_size=0.2, random_state=42
)

# 2. Define Activation Functions
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(a):
    return a * (1 - a)

def softmax(z):
    exp = np.exp(z - np.max(z, axis=1, keepdims=True))
    return exp / np.sum(exp, axis=1, keepdims=True)

# 3. Initialize Neural Network Parameters (4 -> 8 -> 3)
np.random.seed(42)
input_dim = X_train.shape[1]    # 4 features
hidden_dim = 8                  # 8 hidden neurons
output_dim = y_train.shape[1]   # 3 classes
learning_rate = 0.1
epochs = 500

W1 = np.random.randn(input_dim, hidden_dim) * 0.1
b1 = np.zeros((1, hidden_dim))
W2 = np.random.randn(hidden_dim, output_dim) * 0.1
b2 = np.zeros((1, output_dim))

# 4. Train Neural Network with Backpropagation
for epoch in range(epochs):
    # Forward Propagation
    Z1 = np.dot(X_train, W1) + b1
    A1 = sigmoid(Z1)
    Z2 = np.dot(A1, W2) + b2
    A2 = softmax(Z2)

    # Compute Loss & Accuracy
    loss = -np.mean(np.sum(y_train * np.log(A2 + 1e-8), axis=1))
    y_pred_train = np.argmax(A2, axis=1)
    y_true_train = np.argmax(y_train, axis=1)
    acc = np.mean(y_pred_train == y_true_train)

    # Backpropagation
    dZ2 = A2 - y_train
    dW2 = np.dot(A1.T, dZ2) / X_train.shape[0]
    db2 = np.sum(dZ2, axis=0, keepdims=True) / X_train.shape[0]
    dA1 = np.dot(dZ2, W2.T)
    dZ1 = dA1 * sigmoid_derivative(A1)
    dW1 = np.dot(X_train.T, dZ1) / X_train.shape[0]
    db1 = np.sum(dZ1, axis=0, keepdims=True) / X_train.shape[0]

    # Gradient Descent Update
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2

    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1:3d}/{epochs} | Loss: {loss:.4f} | Accuracy: {acc*100:.2f}%")

# 5. Evaluate on Test Set
Z1_test = np.dot(X_test, W1) + b1
A1_test = sigmoid(Z1_test)
Z2_test = np.dot(A1_test, W2) + b2
A2_test = softmax(Z2_test)

y_pred_test = np.argmax(A2_test, axis=1)
y_true_test = np.argmax(y_test, axis=1)
test_accuracy = np.mean(y_pred_test == y_true_test)

print(f"\nFinal Test Accuracy: {test_accuracy*100:.2f}%")
print(f"\nConfusion Matrix:\n{confusion_matrix(y_true_test, y_pred_test)}")
print("\nClassification Report:\n" + classification_report(
    y_true_test, y_pred_test, target_names=iris.target_names
))

Epoch 100/500 | Loss: 0.9274 | Accuracy: 84.17%
Epoch 200/500 | Loss: 0.5574 | Accuracy: 87.50%
Epoch 300/500 | Loss: 0.4303 | Accuracy: 92.50%
Epoch 400/500 | Loss: 0.3596 | Accuracy: 93.33%
Epoch 500/500 | Loss: 0.3076 | Accuracy: 94.17%

Final Test Accuracy: 93.33%

Confusion Matrix:
[[10  0  0]
 [ 0  7  2]
 [ 0  0 11]]

Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      0.78      0.88         9
   virginica       0.85      1.00      0.92        11

    accuracy                           0.93        30
   macro avg       0.95      0.93      0.93        30
weighted avg       0.94      0.93      0.93        30

